# Hardware A/B — Cyclical Drift Detection

Compare two devices (A vs B) where B has a slight period drift (e.g., fan cycle, power ripple).
Detect the drift via spectral peak shift and use the QFT histogram as an interpretable visualization.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.fft import rfft, rfftfreq
from quantum_hybrid_system import PeriodicState

rng = np.random.default_rng(3)
N = 4096
t = np.arange(N)

pa, pb = 60.0, 63.0  # periods
a = 0.4*np.sin(2*np.pi*t/pa) + rng.normal(0, 0.2, size=N)
b = 0.4*np.sin(2*np.pi*t/pb) + rng.normal(0, 0.2, size=N)

def peak_period(x):
    yf = np.abs(rfft(x - x.mean()))
    xf = rfftfreq(N, d=1.0)
    k = np.argmax(yf[1:]) + 1
    f = xf[k]
    return 1/f if f>0 else np.nan

print("A period ~", peak_period(a))
print("B period ~", peak_period(b))

plt.figure()
plt.plot(t[:600], a[:600], label="A")
plt.plot(t[:600], b[:600], label="B")
plt.legend()
plt.title("Signals A vs B (first 600 samples)")
plt.xlabel("t")
plt.ylabel("value")
plt.show()

# QFT pictures
n = 10
Na = PeriodicState(n, period=int(round(peak_period(a))))
Nb = PeriodicState(n, period=int(round(peak_period(b))))
Ha = Na.measure(num_shots=3000, use_qft=True)
Hb = Nb.measure(num_shots=3000, use_qft=True)

def coarse_hist(samples, bins=64):
    N = 2**n
    hist = np.zeros(bins, dtype=int)
    for s in samples:
        hist[(s * bins) // N] += 1
    return hist

ha = coarse_hist(Ha)
hb = coarse_hist(Hb)

plt.figure()
plt.bar(np.arange(64)-0.2, ha, width=0.4, label="A")
plt.bar(np.arange(64)+0.2, hb, width=0.4, label="B")
plt.title("QFT coarse histograms (A vs B)")
plt.xlabel("bin")
plt.ylabel("counts")
plt.legend()
plt.show()